# 01 – Business Understanding & Data Audit

**Project:** AI-Powered Project Planning & Risk Forecasting Platform  
**Team:** certAIn Product Intelligence  

## Project Context

certAIn is an AI-powered decision-support platform designed to improve how projects are planned, evaluated, and monitored.

Traditional project planning tools rely heavily on static artifacts such as spreadsheets and Gantt charts. These approaches assume fixed task durations and do not adequately capture uncertainty, interdependencies, or evolving risk conditions.

As a result, project teams often:
- commit to unrealistic timelines
- underestimate execution risk
- lack visibility into delay drivers
- struggle to communicate uncertainty to stakeholders

certAIn addresses these limitations by combining:
- AI-generated task planning
- dependency-aware workflow modeling (DAG)
- probabilistic schedule forecasting using Monte Carlo simulation
- machine learning-based risk classification

The platform transforms project planning into a **data-driven, probabilistic decision-support workflow**.

## Problem Statement

Project planning in many organizations is still based on deterministic assumptions and manual estimation.

Key limitations include:

- **No uncertainty modeling**  
  Task durations are treated as fixed values rather than probabilistic ranges.

- **Limited dependency awareness**  
  Interactions between tasks are not fully modeled, leading to unrealistic timelines.

- **Reactive risk management**  
  Risks are often identified late rather than predicted early.

- **Lack of decision support**  
  Stakeholders do not have access to scenario comparisons or probability-based forecasts.

These limitations lead to:
- frequent delays
- budget overruns
- low confidence in project plans

The core problem is therefore:

> How to transform project planning from a static, assumption-driven process into a probabilistic, data-driven decision-support system?

## Solution Overview

certAIn introduces an integrated pipeline that combines AI, graph modeling, simulation, and machine learning.

The workflow is structured as follows:

1. A user provides a project description (plain text or structured input)
2. An LLM generates a structured task plan
3. Tasks are validated and converted into a Directed Acyclic Graph (DAG)
4. The critical path is computed based on task dependencies
5. Monte Carlo simulation models uncertainty in task durations
6. A machine learning model assigns task-level risk classifications
7. Results are presented through a dashboard and executive summary

This approach enables:
- probabilistic completion forecasts (P50, P80)
- delay probability estimation
- identification of critical tasks and delay drivers
- comparison of execution scenarios

The system is designed as a **decision-support platform**, not just a predictive model.

## Dataset Strategy

This project uses multiple datasets, each serving a distinct role in the system.

### 1. Task-Level Dataset – `construction_dataset.csv`

- Used for training the machine learning risk classifier
- Contains task-level features such as:
  - duration
  - resource requirements
  - constraints
  - dependency count
- Target variable: `Risk_Level`

This dataset supports **supervised learning for risk prediction**.

---

### 2. Project-Level Dataset – `project_portfolio_history.csv`

- Used for exploratory data analysis (EDA)
- Represents historical project performance
- Includes:
  - delay information
  - project characteristics
  - performance metrics

This dataset is used to:
- understand delay patterns
- inform simulation assumptions
- provide portfolio-level context

---

### 3. Monitoring Dataset – `risk_monitoring_snapshot.csv`

- Used for model and system monitoring
- Includes:
  - prediction confidence
  - forecast error
  - drift indicators
  - alert signals

This dataset supports:
- post-deployment monitoring
- performance tracking
- drift analysis

---

These datasets are **not combined into a single table**.  
Instead, they support different components of the platform.

## Data Structure Overview

A lightweight validation is performed to confirm that datasets are loaded correctly and contain the expected structure.

This step ensures:
- correct dataset dimensions
- expected columns are present
- no immediate structural issues

## Data Quality Assessment

A preliminary audit of the datasets indicates:

- no missing values across key variables
- no duplicate records detected
- consistent data types across columns
- well-structured numerical and categorical features

These observations suggest that the datasets are suitable for further analysis without extensive preprocessing.

## Key Business Questions

This project aims to answer the following:

### Planning & Forecasting
- What is a realistic project completion timeline under uncertainty?
- What is the probability that a project will exceed its deadline?

### Risk Understanding
- Which factors contribute most to project delays?
- Which tasks are most likely to become critical bottlenecks?

### Predictive Modeling
- Can task-level risk be predicted using structured features?
- Which machine learning model performs best for this task?

### Monitoring & Decision Support
- How does model performance evolve over time?
- How drift or degradation are detected in predictions?

These questions guide the design of all subsequent notebooks.

## System Perspective

This capstone should be understood as a **system-level solution**, not just a collection of models.

The platform integrates multiple components:

- AI planning (LLM)
- graph-based workflow modeling (DAG)
- simulation (Monte Carlo)
- machine learning (risk classification)
- monitoring and evaluation

Each component addresses a different aspect of project uncertainty.

Importantly:
- The ML model is **advisory**, not the sole decision engine
- The Monte Carlo simulation provides **probabilistic forecasting**
- The DAG structure ensures **dependency-aware realism**

Together, these form a **holistic decision-support system**.

## Role of This Notebook

This notebook establishes the foundation for the project by:

- defining the business problem
- explaining the system architecture at a high level
- clarifying dataset roles and usage
- framing the key analytical questions

The following notebooks build on this foundation:

- Notebook 02: Exploratory Data Analysis (EDA)
- Notebook 03: Model Training and Comparison
- Notebook 04: DAG-Based Forecasting and Monte Carlo Simulation
- Notebook 05: Monitoring and Evaluation

## Expected Impact

Project delays and cost overruns are common challenges in project management.
By combining machine learning and probabilistic forecasting, this system aims
to provide early visibility into potential project risks.

The tool helps project managers:

• identify high-risk projects earlier  
• forecast realistic completion timelines  
• improve planning decisions using data-driven insights  
• reduce delivery uncertainty and budget overruns

In [20]:
# Core libraries

import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Notebook settings
sns.set_style("whitegrid")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Load datasets

In [21]:
project_df = pd.read_csv("../data/project_portfolio_history.csv")
monitoring_df = pd.read_csv("../data/risk_monitoring_snapshot.csv")
mldataset_df = pd.read_csv("../data/construction_dataset.csv")

print("Datasets loaded successfully")

Datasets loaded successfully


### Load Dataset

**Key Insight**

- The project portfolio dataset contains historical project-level data including schedule, cost, and performance indicators.
- The construction dataset provides task-level data used for training the risk classification model.
- The monitoring dataset enables comparison between forecasted and actual project outcomes.
- Together, these datasets support end-to-end analysis from data exploration to predictive modeling and evaluation.

In [22]:
# Dataset shapes

print("Project Portfolio History Shape:")
print(project_df.shape)

print("\nRisk Monitoring Snapshot Shape:")
print(monitoring_df.shape)

print("\nML Dataset Shape:")
print(mldataset_df.shape)

Project Portfolio History Shape:
(360, 24)

Risk Monitoring Snapshot Shape:
(270, 12)

ML Dataset Shape:
(1300, 10)


## First Look at the Data

The first few rows of each dataset are being inspected to confirm that the data was loaded correctly and to understand the structure of the variables.

In [23]:
project_df.head()

,Project_ID,Portfolio_Segment,Region,Delivery_Model,Planning_Mode,Complexity_Score,Planned_Duration_Days,Actual_Duration_Days,Delay_Days,Delay_Flag,Planned_Budget_USD,Actual_Budget_USD,Budget_Overrun_Pct,Team_Size,Critical_Path_Task_Count,Vendor_Count,Change_Order_Count,Resource_Buffer_Pct,Weather_Risk_Index,Procurement_Risk_Index,Stakeholder_Alignment_Score,Requirements_Volatility_Score,Forecast_Error_Days,Outcome_Risk_Level
0,P001,Public Sector,South,Design-Build,Spreadsheet,69,301,296,-5,On Time,"5,346,606.61","5,749,331.62",7.53,24,12,11,4,8.30,5.00,35.10,68.10,31.70,3,Low
1,P002,Infrastructure,Central,Design-Build,Spreadsheet,63,266,267,1,Delayed,"5,160,994.06","5,788,216.98",12.15,27,11,8,7,12.00,10.30,7.00,60.20,56.60,1,Low
2,P003,Commercial,North,Agile Hybrid,Spreadsheet,68,304,304,0,On Time,"4,543,371.36","5,210,515.82",14.68,18,14,10,4,4.40,40.90,72.50,52.90,43.30,2,Medium
3,P004,Commercial,West,EPC,Spreadsheet,50,244,245,1,Delayed,"3,808,792.38","4,328,783.59",13.65,19,10,9,2,11.20,22.40,52.30,57.70,35.60,1,Low
4,P005,Industrial,South,Design-Build,Manual,61,292,299,7,Delayed,"4,317,590.16","4,697,882.16",8.81,20,10,5,7,2.40,35.20,25.80,53.50,44.20,11,Low


In [24]:
monitoring_df.head()

,Snapshot_Month,Project_ID,Portfolio_Segment,Model_Version,Predicted_Risk_Level,Actual_Risk_Level,Prediction_Confidence,P80_Forecast_Days,Actual_Completion_Days,Absolute_Forecast_Error_Days,Drift_Score,Alerts_Open
0,2025-07-01,P013,Industrial,v1-advisory-multimodel,Low,Low,0.80,314,308,6,0.32,0
1,2025-07-01,P034,Public Sector,v1-advisory-multimodel,Low,Low,0.70,298,260,38,0.41,0
2,2025-07-01,P064,Energy,v1-advisory-multimodel,Low,Low,0.53,293,288,5,0.41,0
3,2025-07-01,P066,Infrastructure,v1-advisory-multimodel,Low,Low,0.69,289,274,15,0.36,0
4,2025-07-01,P069,Industrial,v1-advisory-multimodel,Low,Low,0.64,260,249,11,0.47,0


In [25]:
mldataset_df.head()

,Task_ID,Task_Duration_Days,Labor_Required,Equipment_Units,Material_Cost_USD,Start_Constraint,Risk_Level,Resource_Constraint_Score,Site_Constraint_Score,Dependency_Count
0,T1,52,14,6,"16,789.73",0,Medium,0.41,0.59,4
1,T2,15,2,2,"16,885.80",5,Low,0.75,0.17,3
2,T3,72,11,1,"7,978.70",22,Low,0.96,0.41,1
3,T4,61,1,5,"19,379.02",18,Low,0.41,0.67,4
4,T5,21,19,5,"66,757.72",22,Low,0.85,0.63,3


## Data Types

Understanding data types helps determine:

- which variables are categorical
- which variables are numerical
- which variables can be used for statistical analysis or machine learning models

In [26]:
project_df.dtypes

Project_ID                        object
Portfolio_Segment                 object
Region                            object
Delivery_Model                    object
Planning_Mode                     object
Complexity_Score                   int64
Planned_Duration_Days              int64
Actual_Duration_Days               int64
Delay_Days                         int64
Delay_Flag                        object
Planned_Budget_USD               float64
Actual_Budget_USD                float64
Budget_Overrun_Pct               float64
Team_Size                          int64
Critical_Path_Task_Count           int64
Vendor_Count                       int64
Change_Order_Count                 int64
Resource_Buffer_Pct              float64
Weather_Risk_Index               float64
Procurement_Risk_Index           float64
Stakeholder_Alignment_Score      float64
Requirements_Volatility_Score    float64
Forecast_Error_Days                int64
Outcome_Risk_Level                object
dtype: object

In [27]:
monitoring_df.dtypes

Snapshot_Month                   object
Project_ID                       object
Portfolio_Segment                object
Model_Version                    object
Predicted_Risk_Level             object
Actual_Risk_Level                object
Prediction_Confidence           float64
P80_Forecast_Days                 int64
Actual_Completion_Days            int64
Absolute_Forecast_Error_Days      int64
Drift_Score                     float64
Alerts_Open                       int64
dtype: object

In [28]:
mldataset_df.dtypes

Task_ID                       object
Task_Duration_Days             int64
Labor_Required                 int64
Equipment_Units                int64
Material_Cost_USD            float64
Start_Constraint               int64
Risk_Level                    object
Resource_Constraint_Score    float64
Site_Constraint_Score        float64
Dependency_Count               int64
dtype: object

## Duplicate Records

Duplicate records can distort analysis and model training.

The datasets are checked for duplicate entries to ensure that each project record is unique.

In [32]:
print("Project dataset duplicates:", project_df.duplicated().sum())
print("Monitoring dataset duplicates:", monitoring_df.duplicated().sum())
print("ML dataset duplicates:", mldataset_df.duplicated().sum())

Project dataset duplicates: 0
Monitoring dataset duplicates: 0
ML dataset duplicates: 0


## Basic Dataset Statistics

Summary statistics provide an overview of key numerical variables, including:

- central tendencies
- variability
- potential outliers

These statistics help identify patterns before deeper exploratory analysis.

## Data Structure & Quality

In [33]:
project_df.describe()

,Complexity_Score,Planned_Duration_Days,Actual_Duration_Days,Delay_Days,Planned_Budget_USD,Actual_Budget_USD,Budget_Overrun_Pct,Team_Size,Critical_Path_Task_Count,Vendor_Count,Change_Order_Count,Resource_Buffer_Pct,Weather_Risk_Index,Procurement_Risk_Index,Stakeholder_Alignment_Score,Requirements_Volatility_Score,Forecast_Error_Days
count,360.00,360.00,360.00,360.00,360.00,360.00,360.00,360.00,360.00,360.00,360.00,360.00,360.00,360.00,360.00,360.00,360.00
mean,63.06,276.75,284.61,7.86,"4,549,036.81","5,150,104.81",12.95,21.66,10.26,6.69,3.69,9.25,37.96,45.44,59.82,47.72,15.20
std,11.78,38.76,48.61,17.25,"691,435.66","896,714.86",5.65,4.48,2.28,2.06,1.67,4.23,19.88,20.38,10.17,10.88,12.06
min,27.00,162.00,149.00,-30.00,"2,778,422.58","2,699,848.34",-3.68,11.00,5.00,1.00,0.00,2.00,5.00,7.00,31.70,12.40,1.00
25%,55.00,253.00,253.75,-4.00,"4,063,099.53","4,564,160.97",9.32,19.00,9.00,5.00,3.00,6.10,21.88,29.30,52.30,40.48,6.75
50%,63.00,276.50,282.00,8.00,"4,541,363.75","5,051,349.38",12.68,21.50,10.00,7.00,4.00,9.10,36.95,45.05,59.50,47.35,14.00
75%,71.00,305.00,318.25,19.00,"5,025,255.90","5,749,992.35",17.14,25.00,12.00,8.00,5.00,12.30,52.25,61.73,66.83,55.85,21.00
max,96.00,371.00,427.00,64.00,"6,432,919.71","8,016,129.35",33.43,34.00,16.00,14.00,8.00,22.00,88.40,91.90,96.00,73.60,76.00


In [34]:
monitoring_df.describe()

,Prediction_Confidence,P80_Forecast_Days,Actual_Completion_Days,Absolute_Forecast_Error_Days,Drift_Score,Alerts_Open
count,270.00,270.00,270.00,270.00,270.00,270.00
mean,0.64,300.92,285.11,15.88,0.38,0.04
std,0.12,43.25,48.64,14.09,0.08,0.28
min,0.40,186.00,149.00,0.00,0.20,0.00
25%,0.55,272.00,254.00,7.00,0.33,0.00
50%,0.62,299.00,285.50,11.00,0.38,0.00
75%,0.73,330.00,316.75,19.00,0.43,0.00
max,0.94,439.00,427.00,69.00,0.71,3.00


In [35]:
mldataset_df.describe()

,Task_Duration_Days,Labor_Required,Equipment_Units,Material_Cost_USD,Start_Constraint,Resource_Constraint_Score,Site_Constraint_Score,Dependency_Count
count,"1,300.00","1,300.00","1,300.00","1,300.00","1,300.00","1,300.00","1,300.00","1,300.00"
mean,43.44,10.01,4.98,"49,525.11",14.77,0.59,0.48,2.03
std,25.72,5.45,2.60,"28,409.87",8.59,0.23,0.23,1.42
min,1.00,1.00,1.00,"1,003.04",0.00,0.20,0.10,0.00
25%,21.00,5.00,3.00,"24,176.47",7.00,0.39,0.29,1.00
50%,43.00,10.00,5.00,"49,328.33",15.00,0.59,0.47,2.00
75%,64.00,15.00,7.00,"73,522.53",22.00,0.79,0.67,3.00
max,89.00,19.00,9.00,"99,956.21",29.00,1.00,0.90,4.00


In [38]:
project_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 360 entries, 0 to 359
Data columns (total 24 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Project_ID                     360 non-null    object 
 1   Portfolio_Segment              360 non-null    object 
 2   Region                         360 non-null    object 
 3   Delivery_Model                 360 non-null    object 
 4   Planning_Mode                  360 non-null    object 
 5   Complexity_Score               360 non-null    int64  
 6   Planned_Duration_Days          360 non-null    int64  
 7   Actual_Duration_Days           360 non-null    int64  
 8   Delay_Days                     360 non-null    int64  
 9   Delay_Flag                     360 non-null    object 
 10  Planned_Budget_USD             360 non-null    float64
 11  Actual_Budget_USD              360 non-null    float64
 12  Budget_Overrun_Pct             360 non-null    flo

In [37]:
monitoring_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Snapshot_Month                270 non-null    object 
 1   Project_ID                    270 non-null    object 
 2   Portfolio_Segment             270 non-null    object 
 3   Model_Version                 270 non-null    object 
 4   Predicted_Risk_Level          270 non-null    object 
 5   Actual_Risk_Level             270 non-null    object 
 6   Prediction_Confidence         270 non-null    float64
 7   P80_Forecast_Days             270 non-null    int64  
 8   Actual_Completion_Days        270 non-null    int64  
 9   Absolute_Forecast_Error_Days  270 non-null    int64  
 10  Drift_Score                   270 non-null    float64
 11  Alerts_Open                   270 non-null    int64  
dtypes: float64(2), int64(4), object(6)
memory usage: 25.4+ KB


In [36]:
mldataset_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1300 entries, 0 to 1299
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Task_ID                    1300 non-null   object 
 1   Task_Duration_Days         1300 non-null   int64  
 2   Labor_Required             1300 non-null   int64  
 3   Equipment_Units            1300 non-null   int64  
 4   Material_Cost_USD          1300 non-null   float64
 5   Start_Constraint           1300 non-null   int64  
 6   Risk_Level                 1300 non-null   object 
 7   Resource_Constraint_Score  1300 non-null   float64
 8   Site_Constraint_Score      1300 non-null   float64
 9   Dependency_Count           1300 non-null   int64  
dtypes: float64(3), int64(5), object(2)
memory usage: 101.7+ KB


### Data Structure and Quality

**Key Insight**

- The datasets contain a mix of numerical and categorical features relevant to project risk analysis.
- Data types are correctly assigned, and no major structural issues are observed.
- Initial inspection suggests the datasets are suitable for further analysis and modeling.

In [29]:
project_df.isna().sum()

Project_ID                       0
Portfolio_Segment                0
Region                           0
Delivery_Model                   0
Planning_Mode                    0
Complexity_Score                 0
Planned_Duration_Days            0
Actual_Duration_Days             0
Delay_Days                       0
Delay_Flag                       0
Planned_Budget_USD               0
Actual_Budget_USD                0
Budget_Overrun_Pct               0
Team_Size                        0
Critical_Path_Task_Count         0
Vendor_Count                     0
Change_Order_Count               0
Resource_Buffer_Pct              0
Weather_Risk_Index               0
Procurement_Risk_Index           0
Stakeholder_Alignment_Score      0
Requirements_Volatility_Score    0
Forecast_Error_Days              0
Outcome_Risk_Level               0
dtype: int64

In [30]:
monitoring_df.isna().sum()

Snapshot_Month                  0
Project_ID                      0
Portfolio_Segment               0
Model_Version                   0
Predicted_Risk_Level            0
Actual_Risk_Level               0
Prediction_Confidence           0
P80_Forecast_Days               0
Actual_Completion_Days          0
Absolute_Forecast_Error_Days    0
Drift_Score                     0
Alerts_Open                     0
dtype: int64

In [31]:
mldataset_df.isna().sum()

Task_ID                      0
Task_Duration_Days           0
Labor_Required               0
Equipment_Units              0
Material_Cost_USD            0
Start_Constraint             0
Risk_Level                   0
Resource_Constraint_Score    0
Site_Constraint_Score        0
Dependency_Count             0
dtype: int64

### Missing Values

**Key Insight**

- The datasets show minimal or no missing values across key features.
- This reduces the need for complex imputation strategies.
- Overall data quality is sufficient for reliable downstream analysis.

## Key Insights & Takeaways

Initial observations from the dataset audit:

- The project dataset contains historical project characteristics including schedule, cost, and complexity indicators.
- Key variables such as **Delay_Days**, **Budget_Overrun_Pct**, and **Outcome_Risk_Level** provide useful signals for risk prediction.
- The construction dataset contains **task-level information** such as activity durations, dependencies, and sequencing, which will be used later for **project graph (DAG) construction and Monte Carlo-based schedule forecasting**.
- The monitoring dataset enables evaluation of **forecast reliability and model performance**.
- No major data quality issues were identified during the initial audit.

These datasets appear suitable for further analysis.

The next notebook will perform **Exploratory Data Analysis (EDA)** to identify patterns and relationships between project characteristics and risk outcomes.